You can determine which OCR engine is active in Docling at runtime through three main avenues: inspecting the DocumentConverter format options, checking PdfPipelineOptions, or inspecting the post-conversion metadata.

## Step 1 Inspecting the DocumentConverter Instance

In [3]:
from docling.datamodel.base_models import InputFormat
from docling.document_converter import DocumentConverter

converter = DocumentConverter()

# 1. Access the resolved options dictionary
pdf_options = converter.format_to_options.get(InputFormat.PDF)

# 2. Extract pipeline options and OCR configuration
pipeline_opts = pdf_options.pipeline_options
is_ocr_enabled = getattr(pipeline_opts, "do_ocr", False)
ocr_opts = getattr(pipeline_opts, "ocr_options", None)

print(f"OCR Enabled: {is_ocr_enabled}")
print(f"Configured OCR Class: {type(ocr_opts).__name__ if ocr_opts else 'None'}")
print(f"Configured OCR Details: {ocr_opts}")

OCR Enabled: True
Configured OCR Class: OcrAutoOptions
Configured OCR Details: mode=<OcrMode.DEFAULT: 'default'> lang=[] scale=3.0 force_full_page_ocr=False


## Step 2 Inspecting Default vs. Explicit Configurations
Docling's underlying options classes are typed (e.g., EasyOcrOptions, RapidOcrOptions, TesseractOcrOptions). Checking the class type gives the exact backend being passed

In [4]:
from docling.datamodel.pipeline_options import (
    PdfPipelineOptions,
    EasyOcrOptions,
    RapidOcrOptions,
    TesseractOcrOptions,
)

def identify_ocr_engine(pipeline_options: PdfPipelineOptions) -> str:
    if not pipeline_options.do_ocr:
        return "OCR is disabled (do_ocr=False)"
    
    ocr_opts = pipeline_options.ocr_options
    
    if isinstance(ocr_opts, EasyOcrOptions):
        return f"EasyOCR (Languages: {ocr_opts.lang}, GPU: {getattr(ocr_opts, 'use_gpu', False)})"
    elif isinstance(ocr_opts, RapidOcrOptions):
        return "RapidOCR (ONNX runtime backend)"
    elif isinstance(ocr_opts, TesseractOcrOptions):
        return f"Tesseract (Languages: {ocr_opts.lang})"
    elif ocr_opts is None:
        # Docling defaults to EasyOcrOptions internally when ocr_options is not explicitly passed
        return "Default Engine: EasyOCR (Implicit default)"
    else:
        return f"Custom / Other Engine: {type(ocr_opts).__name__}"

# Test with default pipeline options
default_pipeline = PdfPipelineOptions()
print("Default Setup:", identify_ocr_engine(default_pipeline))

# Test with explicit RapidOCR
custom_pipeline = PdfPipelineOptions()
custom_pipeline.ocr_options = RapidOcrOptions()
print("Custom Setup:", identify_ocr_engine(custom_pipeline))

Default Setup: Custom / Other Engine: OcrAutoOptions
Custom Setup: RapidOCR (ONNX runtime backend)


In [5]:
from docling.datamodel.base_models import InputFormat
from docling.document_converter import DocumentConverter

converter = DocumentConverter()

# Initialize the pipeline backend to inspect the resolved OCR engine
pipeline = converter.initialize_pipeline(InputFormat.PDF)

# Check the OCR engine instantiated by the pipeline
if hasattr(pipeline, "ocr_engine") and pipeline.ocr_engine is not None:
    engine_instance = pipeline.ocr_engine
    print(f"Active OCR Engine: {type(engine_instance).__name__}")
    print(f"Engine Details: {engine_instance}")
else:
    print("OCR is resolved per page or disabled.")

[INFO] 2026-08-22 11:48:43,585 [RapidOCR] base.py:23: Using engine_name: torch
[INFO] 2026-08-22 11:48:43,626 [RapidOCR] device_config.py:57: Using CPU device
[INFO] 2026-08-22 11:48:43,733 [RapidOCR] download_file.py:60: File exists and is valid: D:\AI Learning\rag-learning\.venv\Lib\site-packages\rapidocr\models\PP-OCRv6_det_small.pth
[INFO] 2026-08-22 11:48:43,735 [RapidOCR] main.py:50: Using D:\AI Learning\rag-learning\.venv\Lib\site-packages\rapidocr\models\PP-OCRv6_det_small.pth
[INFO] 2026-08-22 11:48:44,987 [RapidOCR] base.py:23: Using engine_name: torch
[INFO] 2026-08-22 11:48:44,989 [RapidOCR] device_config.py:57: Using CPU device
[INFO] 2026-08-22 11:48:45,006 [RapidOCR] download_file.py:60: File exists and is valid: D:\AI Learning\rag-learning\.venv\Lib\site-packages\rapidocr\models\ch_ptocr_mobile_v2.0_cls_mobile.pth
[INFO] 2026-08-22 11:48:45,008 [RapidOCR] main.py:50: Using D:\AI Learning\rag-learning\.venv\Lib\site-packages\rapidocr\models\ch_ptocr_mobile_v2.0_cls_mobil

OCR is resolved per page or disabled.
